# Бренды: показатель SoS

Пример расчета показателя SoS внутри категории - доли запросов с указанием бренда только среди запросов с выбранными брендами

## Описание задачи и условий расчета
- Период: весь доступный период
- Категории: все доступные категории (в примере "Смартфоны")
- Бренды: бренды доступных категорий
- Статистики: SoS (%)

### Инициализация

Импортируем библиотеки, которые понадобятся для работы.\
Выполните следующую ячейку, для этого перейдите в нее и нажмите Shift + Enter

In [ ]:
import pandas as pd
import numpy as np

from fileapiback import *

### Формирование задания

Зададим условия для расчета.\
*см. подробнее формат условий в блокноте [help.ipynb](help.ipynb)*

In [ ]:
# укажите период в формате:
# None (все месяцы с активностью в архиве) или ['YYYY-MM-01', 'YYYY-MM-01'] (конкретные месяцы)
period = None

# укажите одну категорию брендов для фильтрации в формате: 'Категория'
category_filter = 'Смартфоны'

# укажите бренды из категории для фильтрации в формате:
# None (все бренды) или  ['бренд1', 'бренд2']
brand_filter = None

### Выполнение задания
- Визуализация динамики количества запросов
- Расчет
  - количества запросов в разбивке по брендам по месяцам (тыс)
  - распределения запросов в разбивке по брендам по типу ресурса по месяцам (%)

In [ ]:
# формирование pandas.DataFrame

obj = FileApi()

# создаем pandas.DataFrame
data_filt, data = obj.create_pdDF(note='brand_sos', period=period, 
                                  category_filter=category_filter, brand_filter=brand_filter)

In [ ]:
# расчет SoS
    
qcnt_brand = (data_filt # считаем количество запросов по указанным брендам
        .groupby(['month', 'Category', 'Brand'])['Weight'] # группируем по месяцу, категории
        .sum()  # сумма по группе
        .reset_index(name='QCnt_brand')  # переименование столбца
        .sort_values(by=['month', 'Category', 'QCnt_brand'], ascending=[True, True, False])  # сортировка
    )

qcnt_all = data.groupby(['month', 'Category'])['Weight'].sum().reset_index(name='QCnt_all') # добавляем общую сумму 

qcnt = (qcnt_brand.merge(qcnt_all, on = ['month', 'Category']))
qcnt['SoS_%'] = (qcnt['QCnt_brand'] / qcnt['QCnt_all'] * 100).round(1) # вычисляем процент для отдельного бренда

# выбираем нужные столбцы и сортируем
qcnt = (qcnt[['month', 'Category', 'Brand', 'SoS_%']]
        .sort_values(['month', 'Category', 'SoS_%'], ascending=[True, True, False]))

In [ ]:
# построение графика

obj.draw_diagram(df=qcnt, period=period, col='SoS_%', brand_cat='Brand')

## Экспорт в Excel
По умолчанию файл сохраняется в ту же директорию, где лежат все блокноты

In [ ]:
time_now = datetime.now().strftime('%Y%m%d_%H%M%S')

with pd.ExcelWriter(f'brands-sos-{time_now}.xlsx') as writer:
    qcnt.to_excel(writer, 'SoS', index=False)